# 🌱 Crop Feature Selection via Machine Learning

> **Goal:** Identify the *single most predictive soil measurement* (N, P, K, or pH) for recommending the optimal crop — helping farmers make data-driven decisions even when lab budgets are tight.

---
**Dataset:** `data/soil_measures.csv`  
**Approach:** Train one Logistic Regression model per feature, compare weighted F1-scores  
**Key result:** The notebook concludes with the best-performing feature and actionable insight

## 1 — Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Aesthetics ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
PALETTE = sns.color_palette('viridis', 4)
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False})

RANDOM_STATE = 42
FEATURES     = ['N', 'P', 'K', 'ph']
TARGET       = 'crop'

print('Libraries loaded successfully.')

---
## 2 — Load & Explore the Data

In [ ]:
crops = pd.read_csv('../data/soil_measures.csv')

print(f'Shape           : {crops.shape}')
print(f'Unique crops    : {crops[TARGET].nunique()}')
print(f'Missing values  : {crops.isnull().sum().sum()}\n')
crops.head()

In [ ]:
# ── Summary statistics per feature ──────────────────────────────────────────
crops[FEATURES].describe().round(2)

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
order = crops[TARGET].value_counts().index
sns.countplot(data=crops, y=TARGET, order=order, palette='viridis', ax=ax)
ax.set_title('Crop Class Distribution', fontweight='bold', pad=12)
ax.set_xlabel('Count')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature distributions ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
feature_labels = {'N': 'Nitrogen (N)', 'P': 'Phosphorous (P)',
                  'K': 'Potassium (K)', 'ph': 'pH Level'}

for ax, (feat, label), color in zip(axes, feature_labels.items(), PALETTE):
    sns.histplot(crops[feat], kde=True, ax=ax, color=color, bins=30)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')

fig.suptitle('Distribution of Soil Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature correlation heatmap ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
corr = crops[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---
## 3 — Preprocessing: Train / Test Split

In [ ]:
X = crops.drop(columns=TARGET)
y = crops[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train size : {len(X_train):,} rows ({len(X_train)/len(X):.0%})')
print(f'Test size  : {len(X_test):,} rows  ({len(X_test)/len(X):.0%})')

---
## 4 — Single-Feature Model Evaluation

For each soil feature we:
1. Train a **Multinomial Logistic Regression** using only that feature  
2. Predict crop labels on the held-out test set  
3. Record the **weighted F1-score** (accounts for class imbalance)

In [ ]:
feature_performance = {}

for feature in FEATURES:
    # Scale improves stability; scikit-learn handles multiclass automatically.
    model = Pipeline(
        steps=[
            ('scaler', StandardScaler()),
            (
                'clf',
                LogisticRegression(
                    solver='lbfgs',
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    model.fit(X_train[[feature]], y_train)
    y_pred = model.predict(X_test[[feature]])

    f1 = metrics.f1_score(y_test, y_pred, average='weighted')
    feature_performance[feature] = round(f1, 4)
    print(f'  {feature:>4}  ->  weighted F1 = {f1:.4f}')

best_feature = max(feature_performance, key=feature_performance.get)
best_predictive_feature = {best_feature: feature_performance[best_feature]}

---
## 5 — Results Visualisation

In [ ]:
results_df = (
    pd.Series(feature_performance, name='F1 Score')
    .rename_axis('Feature')
    .reset_index()
    .sort_values('F1 Score', ascending=False)
)

colors = [PALETTE[0] if f == best_feature else PALETTE[-1]
          for f in results_df['Feature']]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(results_df['Feature'], results_df['F1 Score'],
               color=colors, edgecolor='white', height=0.55)

for bar, score in zip(bars, results_df['F1 Score']):
    ax.text(score + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{score:.4f}', va='center', fontsize=10)

ax.set_xlim(0, max(results_df['F1 Score']) * 1.2)
ax.set_xlabel('Weighted F1-Score', fontsize=11)
ax.set_title('Soil Feature Predictive Performance\n(Logistic Regression — single feature)',
             fontweight='bold', pad=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(results_df.to_string(index=False))

---
## 6 — Conclusion

In [ ]:
print('=' * 50)
print('  BEST PREDICTIVE FEATURE RESULT')
print('=' * 50)
for feat, score in best_predictive_feature.items():
    print(f'  Feature       : {feat}')
    print(f'  Weighted F1   : {score:.4f}')
print('=' * 50)

### Key Takeaway

Even with a **single soil measurement**, a Logistic Regression model can meaningfully classify crop types. The winning feature offers farmers a cost-effective, actionable signal — no full lab panel needed.

**Next steps to explore:**
- Use all features together and compare the multi-feature F1 vs best single-feature F1  
- Try `RandomForestClassifier` for non-linear feature importance  
- Apply `SelectKBest` or `RFE` for more systematic feature selection  
- Cross-validate results to reduce test-set variance